Tratamento dos Dados (Camada Trusted)

Nesta etapa, os dados da camada RAW são lidos e preparados para uso analítico.

O tratamento realizado foi:

**Bairros**
- Validação dos tipos das colunas
- Verificação de possíveis áreas iguais a zero

**Concorrentes**
- Conversão de `codigo_bairro` para tipo inteiro anulável (Int64)
- Garantia de tipagem correta das colunas

**Eventos**
- Remoção de registros duplicados (chave composta: cliente + concorrente + datetime)
- Conversão da coluna `datetime` para tipo datetime
- Padronização dos tipos das colunas

**População**
- Remoção de registros com população nula
- Conversão da coluna `populacao` para inteiro

Após o tratamento, os dados são armazenados na camada `trusted_silver` em formato Parquet.

LER DO RAW_BRONZE

In [1]:
from pathlib import Path
import pandas as pd

RAW_PATH = Path("../data/raw_bronze")

df_bairros = pd.read_csv(RAW_PATH / "bairros.csv")
df_conc = pd.read_csv(RAW_PATH / "concorrentes.csv")
df_eventos = pd.read_csv(RAW_PATH / "eventos_de_fluxo.csv")
df_pop = pd.read_json(RAW_PATH / "populacao.json")

BAIRROS

In [15]:
#Validação e garantia da tipagem e Verificação da area 0
def tratar_bairros(df_bairros):
    print("="*60)
    print("VALIDAÇÃO E PADRONIZAÇÃO - BAIRROS")
    print("="*60)

    # Estado inicial
    print("\nTipos antes:")
    print(df_bairros.dtypes)

    # =========================
    # VALIDAÇÕES
    # =========================

    # Verificar área zero
    area_zero = (df_bairros["area"] == 0).sum()
    print(f"\nBairros com área igual a zero: {area_zero}")

    # Verificar duplicidade da chave
    chave_duplicada = df_bairros["codigo"].duplicated().sum()
    print(f"Códigos duplicados: {chave_duplicada}")

    # =========================
    # PADRONIZAÇÃO DE TIPOS
    # =========================

    df_bairros["codigo"] = df_bairros["codigo"].astype(int)
    df_bairros["area"] = df_bairros["area"].astype(float)
    df_bairros["municipio"] = df_bairros["municipio"].astype(str)
    df_bairros["uf"] = df_bairros["uf"].astype(str)

    # =========================
    # PRINTS FINAIS
    # =========================

    print("\nTipos depois:")
    print(df_bairros.dtypes)

    print("\nResumo final:")
    print(f"Total de registros: {df_bairros.shape[0]}")
    print(f"Área zero encontrada? {'Sim' if area_zero > 0 else 'Não'}")
    print(f"Chave duplicada encontrada? {'Sim' if chave_duplicada > 0 else 'Não'}")

    return df_bairros

df_bairros = tratar_bairros(df_bairros)

VALIDAÇÃO E PADRONIZAÇÃO - BAIRROS

Tipos antes:
codigo         int64
nome          object
municipio     object
uf            object
area         float64
dtype: object

Bairros com área igual a zero: 0
Códigos duplicados: 0

Tipos depois:
codigo         int64
nome          object
municipio     object
uf            object
area         float64
dtype: object

Resumo final:
Total de registros: 133
Área zero encontrada? Não
Chave duplicada encontrada? Não


CONCORRENTES

In [18]:
def tratar_concorrentes_tipagem(df_conc):
    print("="*60)
    print("VALIDAÇÃO E PADRONIZAÇÃO - CONCORRENTES")
    print("="*60)

    # =========================
    # codigo_bairro
    # =========================
    print("\n[ codigo_bairro ]")
    print("Tipo antes:", df_conc["codigo_bairro"].dtype)

    try:
        df_conc["codigo_bairro"] = df_conc["codigo_bairro"].astype("Int64")
        print("Tipo depois:", df_conc["codigo_bairro"].dtype)
    except Exception as e:
        print("Erro ao converter codigo_bairro:", e)

    null_bairro = df_conc["codigo_bairro"].isnull().sum()
    print("Quantidade de null em codigo_bairro:", null_bairro)

    # =========================
    # faixa_preco
    # =========================
    print("\n[ faixa_preco ]")
    print("Tipo atual:", df_conc["faixa_preco"].dtype)

    valores_unicos = df_conc["faixa_preco"].unique()
    print("Valores únicos:", valores_unicos)

    # Validação se é numérico
    is_numeric = pd.api.types.is_numeric_dtype(df_conc["faixa_preco"])
    print("É numérico válido?", is_numeric)

    if not is_numeric:
        print("Faixa_preco não é numérico!")

    print("\nTotal de registros:", df_conc.shape[0])

    return df_conc
df_conc = tratar_concorrentes_tipagem(df_conc)

VALIDAÇÃO E PADRONIZAÇÃO - CONCORRENTES

[ codigo_bairro ]
Tipo antes: Int64
Tipo depois: Int64
Quantidade de null em codigo_bairro: 2752

[ faixa_preco ]
Tipo atual: int64
Valores únicos: [2 0 3 1 4]
É numérico válido? True

Total de registros: 4202


EVENTOS

In [9]:
#Remover duplicados da chave composta
def remover_duplicados_validado(df, subset, mostrar_duplicados=True):
    print(" VALIDAÇÃO DE DUPLICADOS ".center(70, "-"))
    
    total_antes = df.shape[0]
    
    # Encontrar duplicados
    duplicados = df[df.duplicated(subset=subset, keep=False)]
    qtd_duplicados = duplicados.shape[0]
    
    print(f"Total antes: {total_antes:,}")
    print(f"Registros duplicados encontrados: {qtd_duplicados:,}")
    
    if mostrar_duplicados and qtd_duplicados > 0:
        print("\nDuplicados identificados:")
        display(duplicados.sort_values(subset))
    
    # Remover duplicados
    df_limpo = df.drop_duplicates(subset=subset)
    
    total_depois = df_limpo.shape[0]
    removidos = total_antes - total_depois
    
    print(f"\nTotal depois: {total_depois:,}")
    print(f"Registros removidos: {removidos:,}")
    
    print("-" * 70)
    
    return df_limpo

df_eventos = remover_duplicados_validado(
    df_eventos,
    subset=["codigo", "codigo_concorrente", "datetime"]
)

---------------------- VALIDAÇÃO DE DUPLICADOS -----------------------
Total antes: 248,589
Registros duplicados encontrados: 77,561

Duplicados identificados:


,codigo,datetime,codigo_concorrente
181487,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 12:47:25.353,881829878586104
181496,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 12:47:25.353,881829878586104
181950,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 13:37:25.502,881829878586104
181986,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 13:37:25.502,881829878586104
50179,++TPZAjdD0bRDGncsLVDQ2xWxyMkxZvWRBKw7Nr90nIATL...,2017-07-01 11:39:50.926,283641161989443
...,...,...,...
23333,zzHs1b3AH89VWQZhwV256WHkCrwrWzqSRYJUR2TttJ2Xf6...,2017-07-10 21:50:11.711,385915771488086
126447,zzuX/okPMlxvCtvI4vPLV96crTQFAxmR9uGm+M4wYZjgiY...,2017-07-08 21:12:00.341,1406788766202498
126449,zzuX/okPMlxvCtvI4vPLV96crTQFAxmR9uGm+M4wYZjgiY...,2017-07-08 21:12:00.341,1406788766202498
204514,zzuX/okPMlxvCtvI4vPLV96crTQFAxmR9uGm+M4wYZjgiY...,2017-07-08 21:12:14.278,1722354398027676



Total depois: 209,801
Registros removidos: 38,788
----------------------------------------------------------------------


In [ ]:
#Conversão de datetime

print("="*70)
print("CONVERSÃO E VALIDAÇÃO - DATETIME")
print("="*70)

# Tipo antes
print("\nTipo antes:", df_eventos["datetime"].dtype)

# Conversão
df_eventos["datetime"] = pd.to_datetime(
    df_eventos["datetime"],
    format="ISO8601",
    errors="coerce"
)

# Tipo depois
print("Tipo depois:", df_eventos["datetime"].dtype)

# Quantidade de valores inválidos
nat = df_eventos["datetime"].isna().sum()
print("Valores NaT após conversão:", nat)

# Intervalo de datas
if nat < len(df_eventos):
    print("Data mínima:", df_eventos["datetime"].min())
    print("Data máxima:", df_eventos["datetime"].max())
else:
    print("Todas as datas ficaram inválidas!")

print("\nAmostra:")
display(df_eventos["datetime"].head())

CONVERSÃO E VALIDAÇÃO - DATETIME

Tipo antes: datetime64[ns]
Tipo depois: datetime64[ns]
Valores NaT após conversão: 0
Data mínima: 2017-05-31 22:34:36.923000
Data máxima: 2017-07-31 20:51:01.189000

Amostra:


0   2017-07-27 09:51:02.000
1   2017-06-24 14:00:26.405
2   2017-07-06 21:51:11.056
3   2017-07-02 14:27:09.316
4   2017-07-15 03:03:29.731
Name: datetime, dtype: datetime64[ns]

In [13]:
#Garantia de tipos

print(" VALIDAÇÃO DE TIPOS ".center(70, "-"))

tipos_esperados = {
    "codigo": "object",
    "codigo_concorrente": "int64",
    "datetime": "datetime64[ns]"
}

for coluna, tipo_esperado in tipos_esperados.items():
    tipo_atual = str(df_eventos[coluna].dtype)
    status = "OK" if tipo_atual == tipo_esperado else "ERRO"
    print(f"{coluna:<20} -> {tipo_atual:<20} [{status}]")

------------------------- VALIDAÇÃO DE TIPOS -------------------------
codigo               -> object               [OK]
codigo_concorrente   -> int64                [OK]
datetime             -> datetime64[ns]       [OK]


POPULAÇÃO

In [14]:
#Remover null e Converter para int
def tratar_populacao(df_pop):
    print("="*60)
    print("ESTADO INICIAL")
    print("="*60)
    
    print("\nShape inicial:", df_pop.shape)
    print("\nTipos antes:")
    print(df_pop.dtypes)
    
    print("\nNulls antes:")
    print(df_pop.isnull().sum())
    
    print("\nValores únicos de populacao antes:")
    print(df_pop["populacao"].unique()[:10])
    
    
    # ==========================
    # TRATAMENTO
    # ==========================
    
    df_tratado = df_pop.copy()
    
    # Remover null
    df_tratado = df_tratado.dropna(subset=["populacao"])
    
    # Converter para int
    df_tratado["populacao"] = df_tratado["populacao"].astype(int)
    
    
    print("\n" + "="*60)
    print("ESTADO APÓS TRATAMENTO")
    print("="*60)
    
    print("\nShape final:", df_tratado.shape)
    
    print("\nTipos depois:")
    print(df_tratado.dtypes)
    
    print("\nNulls depois:")
    print(df_tratado.isnull().sum())
    
    print("\nValores únicos de populacao depois:")
    print(df_tratado["populacao"].unique()[:10])
    
    return df_tratado

df_pop = tratar_populacao(df_pop)

ESTADO INICIAL

Shape inicial: (133, 2)

Tipos antes:
codigo         int64
populacao    float64
dtype: object

Nulls antes:
codigo       0
populacao    1
dtype: int64

Valores únicos de populacao antes:
[  8717.   5764.   1195.  17840.   1252.    206.   1809. 105720.   2334.
   1064.]

ESTADO APÓS TRATAMENTO

Shape final: (132, 2)

Tipos depois:
codigo       int64
populacao    int64
dtype: object

Nulls depois:
codigo       0
populacao    0
dtype: int64

Valores únicos de populacao depois:
[  8717   5764   1195  17840   1252    206   1809 105720   2334   1064]


SALVANDO EM PARQUET

In [22]:
print(" SALVANDO EM PARQUET ".center(70, "-"))

# Definir caminho da camada trusted
TRUSTED_PATH = Path("../data/trusted_silver")

# Criar pasta se não existir
TRUSTED_PATH.mkdir(parents=True, exist_ok=True)

# Salvar arquivos
df_bairros.to_parquet(TRUSTED_PATH / "bairros.parquet", index=False)
df_conc.to_parquet(TRUSTED_PATH / "concorrentes.parquet", index=False)
df_eventos.to_parquet(TRUSTED_PATH / "eventos.parquet", index=False)
df_pop.to_parquet(TRUSTED_PATH / "populacao.parquet", index=False)

print("Arquivos salvos com sucesso em:", TRUSTED_PATH)

------------------------ SALVANDO EM PARQUET -------------------------
Arquivos salvos com sucesso em: ../data/trusted_silver


Código para testar leitura da TRUSTED

In [23]:
print(" TESTE DE LEITURA - TRUSTED_SILVER ".center(70, "-"))

TRUSTED_PATH = Path("../data/trusted_silver")

# Ler arquivos
bairros = pd.read_parquet(TRUSTED_PATH / "bairros.parquet")
concorrentes = pd.read_parquet(TRUSTED_PATH / "concorrentes.parquet")
eventos = pd.read_parquet(TRUSTED_PATH / "eventos.parquet")
populacao = pd.read_parquet(TRUSTED_PATH / "populacao.parquet")

datasets = {
    "BAIRROS": bairros,
    "CONCORRENTES": concorrentes,
    "EVENTOS": eventos,
    "POPULACAO": populacao
}

for nome, df in datasets.items():
    print("\n" + f" {nome} ".center(70, "-"))
    print("Shape:", df.shape)
    print("Tipos:")
    print(df.dtypes)
    print("\nTop 5:")
    display(df.head())

----------------- TESTE DE LEITURA - TRUSTED_SILVER ------------------

------------------------------ BAIRROS -------------------------------
Shape: (133, 5)
Tipos:
codigo         int64
nome          object
municipio     object
uf            object
area         float64
dtype: object

Top 5:


,codigo,nome,municipio,uf,area
0,355620110,Observatório,Valinhos,SP,68.000900
1,3519071024,Rp 6-24,Hortolândia,SP,0.981768
2,3536505002,Jardim De Itapoan,Paulínia,SP,0.808537
3,3519071026,Rp 6-26,Hortolândia,SP,2.211080
4,3536505001,Nova Paulínia,Paulínia,SP,0.386199



---------------------------- CONCORRENTES ----------------------------
Shape: (4202, 8)
Tipos:
codigo            int64
nome             object
categoria        object
faixa_preco       int64
endereco         object
municipio        object
uf               object
codigo_bairro     Int64
dtype: object

Top 5:


,codigo,nome,categoria,faixa_preco,endereco,municipio,uf,codigo_bairro
0,431962533652067,Boizão Lanches,Bar,2,13190-000 Monte Mor,Monte Mor,SP,<NA>
1,1663855903830869,Bar do Serjão,Bar,0,"Rua das Dracenas, Americana",Americana,SP,<NA>
2,567824576564110,Recanto Do Kuca,Restaurant,0,Jarinu,Jarinu,SP,<NA>
3,202740866540615,Dedé Abelhuda,Grocery Store,0,"SP, 13150000 Cosmópolis",Cosmópolis,SP,<NA>
4,1784900838394305,Tenshi Sushi Boteco Itu,Sushi Restaurant,3,"Av. Plaza, 170, 13302-100 Itu",Itu,SP,<NA>



------------------------------ EVENTOS -------------------------------
Shape: (209801, 3)
Tipos:
codigo                        object
datetime              datetime64[ns]
codigo_concorrente             int64
dtype: object

Top 5:


,codigo,datetime,codigo_concorrente
0,oMn07h1bJYV0Wdx+RTzsDcT8JQlT7QXc7q8A/4y+cO5gBQ...,2017-07-27 09:51:02.000,650509405109544
1,iZuQeTd9am+qfaiqnn5kkixogIbwN0nY2gtMwZqH9bFqph...,2017-06-24 14:00:26.405,650509405109544
2,iIMvwWSnQoW0aqQlcvjc8A6LvjkRX1HLppdkdQZapPVVv7...,2017-07-06 21:51:11.056,650509405109544
3,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-02 14:27:09.316,650509405109544
4,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-15 03:03:29.731,650509405109544



----------------------------- POPULACAO ------------------------------
Shape: (132, 2)
Tipos:
codigo       int64
populacao    int64
dtype: object

Top 5:


,codigo,populacao
0,355620110,8717
1,3519071024,5764
2,3536505002,1195
3,3519071026,17840
4,3536505001,1252
